[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/ch08_01_rnn_from_scratch.ipynb)

# Chapter 8-1: RNN(순환 신경망, Recurrent Neural Network) 밑바닥부터 구현하기

> **목표**: RNN의 수학적 원리를 완전히 이해하고, NumPy로 직접 구현한 뒤, PyTorch와 비교합니다.
>
> **대상**: 딥러닝을 처음 공부하는 분 (선형대수 기초만 알면 됩니다)
>
> **소요 시간**: 약 2~3시간

---

## 📋 목차

| 파트 | 내용 | 핵심 |
|------|------|------|
| **Part 1** | RNN 이론 | 수식 완전 분해, BPTT(시간 역전파) 유도 |
| **Part 2** | NumPy 밑바닥 구현 | RNNCell 클래스, 사인파 예측, 기울기 소실 시각화 |
| **Part 3** | PyTorch nn.RNN | 감정 분류 예제 (Many-to-One, 다대일) |
| **Part 4** | 정리 | 한계점, LSTM 동기, 수식 요약 |

### 학습 흐름도

```
Part 1: 이론                    Part 2: 구현                    Part 3: PyTorch
┌──────────────────┐           ┌──────────────────┐           ┌──────────────────┐
│ 1.1 왜 RNN?      │           │ 2.1 RNNCell 구현 │           │ 3.1 nn.RNN 사용  │
│ 1.2 핵심 수식    │──구현──→  │ 2.2 순전파 검증  │──비교──→  │ 3.2 감정 분류    │
│ 1.3 순전파       │           │ 2.3 역전파 검증  │           └──────────────────┘
│ 1.4 BPTT 역전파  │           │ 2.4 사인파 예측  │                    │
│ 1.5 구조 변형    │           │ 2.5 기울기 소실  │                    ▼
└──────────────────┘           └──────────────────┘           ┌──────────────────┐
                                                              │ Part 4: 정리     │
                                                              │ 한계 → LSTM 동기 │
                                                              └──────────────────┘
```

In [ ]:
# ============================================================
# Colab 환경 설정 셀
# Google Colab에서 실행할 때 필요한 패키지를 설치합니다.
# 로컬 환경에서는 이미 설치되어 있다면 건너뛰어도 됩니다.
# ============================================================

# Colab 환경인지 자동 감지하여 필요한 경우에만 설치
# (왜? 로컬에서 !pip install을 실행하면 불필요한 재설치가 발생할 수 있음)
import subprocess, sys

try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    # numpy: 행렬 연산을 위한 핵심 라이브러리 (RNN 밑바닥 구현에 사용)
    # matplotlib: 기울기 소실 등을 시각화하기 위한 그래프 라이브러리
    # torch: PyTorch - Part 3에서 nn.RNN과 비교할 때 사용
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "numpy", "matplotlib", "torch"])
    # Colab에서 한글 폰트 설치 (그래프에 한글이 깨지지 않도록)
    subprocess.check_call(["apt-get", "-qq", "install", "-y", "fonts-nanum"])
    import matplotlib
    matplotlib.font_manager._rebuild()
    print("Colab 환경: 패키지 및 한글 폰트 설치 완료!")
else:
    print("로컬 환경: 패키지가 이미 설치되어 있다고 가정합니다.")

In [ ]:
# ============================================================
# 공통 임포트(import)
# 이 노트북 전체에서 사용할 라이브러리를 미리 불러옵니다.
# ============================================================

import numpy as np                    # 행렬/벡터 연산 (밑바닥 구현의 핵심)
import matplotlib.pyplot as plt        # 그래프 시각화
import matplotlib                      # 한글 폰트 설정용

# --- 한글 폰트 설정 (그래프에 한글이 깨지지 않도록) ---
# macOS의 경우 AppleGothic, Colab/Linux의 경우 NanumGothic 사용
import platform  # 운영체제를 확인하기 위한 모듈
if platform.system() == 'Darwin':      # macOS인 경우
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                  # Colab(Linux) 등의 경우
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호가 깨지지 않도록 설정

# 난수 시드(seed) 고정: 실행할 때마다 같은 결과를 얻기 위해
# (왜? 랜덤 초기화가 달라지면 매번 다른 결과가 나와서 디버깅이 어려움)
np.random.seed(42)

print("모든 라이브러리가 성공적으로 로드되었습니다!")

---

# Part 1: RNN 이론 - 수식 완전 분해

---

## 1.1 RNN이 왜 필요한가? — 시퀀스(순서가 있는) 데이터의 특성

### 일반 신경망(MLP, 다층 퍼셉트론)의 한계

일반적인 다층 퍼셉트론(MLP, Multi-Layer Perceptron)은 **고정 크기 입력**을 받아 **고정 크기 출력**을 냅니다.

$$
\mathbf{y} = f(\mathbf{W}\mathbf{x} + \mathbf{b})
$$

> 쉽게 말하면, MLP는 "사진 한 장을 보고 판단하는 것"과 비슷합니다. 한 번에 하나의 입력만 보고 답을 내죠.

하지만 현실 세계의 많은 데이터는 **순서(시퀀스, sequence)**가 중요합니다:

| 데이터 유형 | 예시 | 순서가 중요한 이유 |
|---|---|---|
| 자연어 | "나는 밥을 먹었다" | 단어 순서가 바뀌면 의미가 달라짐 |
| 시계열 | 주가, 온도 | 과거 값이 미래에 영향 |
| 음성 | 음파 신호 | 시간 순서대로 처리해야 인식 가능 |
| DNA | ATCG 서열 | 순서가 유전 정보를 결정 |

### 핵심 문제: "기억(Memory)"이 필요하다

"나는 **프랑스**에서 자랐다. ... 나는 **프랑스어**를 잘한다."

이 문장을 이해하려면 앞부분의 "프랑스"를 **기억**하고 있어야 뒷부분의 "프랑스어"를 예측할 수 있습니다. MLP는 각 입력을 **독립적으로** 처리하므로 이런 맥락(context, 문맥)을 유지할 수 없습니다.

> 비유하면, MLP는 **금붕어**처럼 매 순간 기억을 리셋하고, RNN은 **일기장**을 쓰면서 과거를 참고하는 것과 같습니다.

### RNN의 핵심 아이디어

> **이전 시점의 정보를 "은닉 상태(hidden state, 숨겨진 기억)"에 저장하고, 다음 시점으로 전달한다.**

```
시점 t=0        시점 t=1        시점 t=2
  x₀               x₁               x₂          ← 입력 (예: 단어)
  ↓                ↓                ↓
┌─────┐  h₀→  ┌─────┐  h₁→  ┌─────┐
│ RNN │───────→│ RNN │───────→│ RNN │──→ h₂     ← 은닉 상태 (기억) 전달
└─────┘        └─────┘        └─────┘
  ↓                ↓                ↓
  y₀               y₁               y₂          ← 출력 (예: 다음 단어 예측)
```

**같은 RNN 셀**이 매 시점마다 반복(recurrent)되면서, 이전 시점의 은닉 상태 $h_{t-1}$을 다음 시점으로 넘겨줍니다. 이것이 "순환(Recurrent)"이라는 이름의 유래입니다.

> 쉽게 말하면, RNN은 **릴레이 경주**와 비슷합니다. 각 주자(시점)가 바통(은닉 상태)을 다음 주자에게 전달하면서 정보를 이어갑니다.

## 1.2 RNN 핵심 수식 완전 분해

### 두 개의 핵심 수식

RNN의 동작은 단 **두 개의 수식**으로 완전히 설명됩니다:

$$
\boxed{h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)} \quad \cdots \text{(수식 1: 은닉 상태 업데이트)}
$$

$$
\boxed{y_t = W_{hy} \cdot h_t + b_y} \quad \cdots \text{(수식 2: 출력 계산)}
$$

> **수식 1을 쉽게 말하면**: "현재 입력($x_t$)과 이전 기억($h_{t-1}$)을 적절히 섞어서 새로운 기억($h_t$)을 만든다."
> 마치 오늘의 일기를 쓸 때, 오늘 있었던 일(입력)과 어제까지의 기억(이전 은닉 상태)을 합쳐서 새로운 기억을 만드는 것과 같습니다.
>
> **수식 2를 쉽게 말하면**: "현재 기억($h_t$)을 바탕으로 답($y_t$)을 내놓는다."

### 각 기호의 의미와 차원

아래 표에서 모든 변수의 의미를 정리합니다. **차원을 이해하는 것이 가장 중요합니다.**

설정:
- $d$: 입력 벡터의 차원 (예: 단어 임베딩(embedding, 단어를 숫자 벡터로 변환한 것) 크기 = 10)
- $h$: 은닉 상태의 차원 (예: 64, 128 — 우리가 직접 정하는 하이퍼파라미터(초매개변수))
- $o$: 출력 벡터의 차원 (예: 분류 클래스 수 = 5)

| 기호 | 이름 | 차원 | 의미 |
|------|------|------|------|
| $x_t$ | 입력 벡터 | $(d, 1)$ | 시점 $t$에서의 입력 (예: 단어 임베딩) |
| $h_t$ | 은닉 상태 | $(h, 1)$ | 시점 $t$에서의 "기억" — 과거 정보의 요약 |
| $h_{t-1}$ | 이전 은닉 상태 | $(h, 1)$ | 바로 이전 시점의 기억 |
| $y_t$ | 출력 벡터 | $(o, 1)$ | 시점 $t$에서의 예측 출력 |
| $W_{xh}$ | 입력→은닉 가중치 | $(h, d)$ | 입력을 은닉 공간으로 변환하는 행렬 |
| $W_{hh}$ | 은닉→은닉 가중치 | $(h, h)$ | 이전 은닉 상태를 현재로 전달하는 행렬 |
| $W_{hy}$ | 은닉→출력 가중치 | $(o, h)$ | 은닉 상태를 출력으로 변환하는 행렬 |
| $b_h$ | 은닉 편향(bias) | $(h, 1)$ | 은닉 상태 계산의 편향 |
| $b_y$ | 출력 편향(bias) | $(o, 1)$ | 출력 계산의 편향 |
| $\tanh$ | 활성화 함수 | — | 값을 $[-1, 1]$ 범위로 압축 |

### 차원 검증 (수식 1)

수식 1의 행렬 곱을 차원으로 검증해봅시다:

$$
\underbrace{W_{xh}}_{(h, d)} \cdot \underbrace{x_t}_{(d, 1)} = (h, 1) \quad \checkmark
$$

$$
\underbrace{W_{hh}}_{(h, h)} \cdot \underbrace{h_{t-1}}_{(h, 1)} = (h, 1) \quad \checkmark
$$

$$
\underbrace{(h, 1)}_{W_{xh} x_t} + \underbrace{(h, 1)}_{W_{hh} h_{t-1}} + \underbrace{(h, 1)}_{b_h} = (h, 1) \quad \checkmark
$$

> 쉽게 말하면, 행렬 곱의 차원 규칙은 "앞 행렬의 열 수 = 뒤 행렬의 행 수"이고, 결과는 "(앞 행렬의 행 수, 뒤 행렬의 열 수)"입니다.

$\tanh$는 원소별(element-wise, 각 원소에 독립적으로) 연산이므로 차원이 변하지 않습니다. 따라서 $h_t$의 차원은 $(h, 1)$ 입니다.

### 차원 검증 (수식 2)

$$
\underbrace{W_{hy}}_{(o, h)} \cdot \underbrace{h_t}_{(h, 1)} + \underbrace{b_y}_{(o, 1)} = (o, 1) \quad \checkmark
$$

출력 $y_t$의 차원은 $(o, 1)$ 입니다.

### 가중치 초기화(initialization) 방법

| 가중치 | 추천 초기화 | 이유 |
|--------|-----------|------|
| $W_{xh}$ | Xavier/Glorot 초기화: $\mathcal{N}(0, \sqrt{2/(d+h)})$ | 입력→은닉 변환의 분산(variance)을 적절히 유지 |
| $W_{hh}$ | 직교 초기화(Orthogonal) | 은닉→은닉은 반복 곱셈되므로, 직교 행렬로 기울기 소실/폭발 방지 |
| $W_{hy}$ | Xavier/Glorot 초기화: $\mathcal{N}(0, \sqrt{2/(h+o)})$ | 은닉→출력 변환 |
| $b_h, b_y$ | 영벡터(zeros) | 편향은 0으로 시작하는 것이 일반적 |

> **Xavier 초기화를 쉽게 말하면**: 가중치를 너무 크지도 너무 작지도 않게 초기화하는 방법입니다. 마치 라디오 볼륨을 적절한 크기로 맞추는 것처럼, 신호가 너무 커지거나 사라지지 않도록 합니다.

### tanh 활성화 함수를 쓰는 이유

$$
\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}
$$

> **tanh를 쉽게 말하면**: 어떤 값이든 $-1$에서 $1$ 사이로 "압축"하는 함수입니다. 마치 시험 점수가 아무리 높아도 100점, 아무리 낮아도 0점으로 제한하는 것처럼, 은닉 상태가 너무 커지거나 작아지는 것을 방지합니다.

- **출력 범위**: $[-1, 1]$ → 은닉 상태가 발산하지 않도록 제한
- **0 중심**: sigmoid($[0,1]$)와 달리 출력이 0을 중심으로 분포 → 학습이 더 안정적
- **미분(도함수)**: $\tanh'(z) = 1 - \tanh^2(z)$ → 최대값이 1 (sigmoid의 최대 0.25보다 큼) → 기울기 소실(gradient vanishing)이 덜함

```
tanh vs sigmoid 비교:

        tanh                          sigmoid
   1 ┤ ........─────────         1 ┤ ........─────────
     │       /                      │       /
   0 ┤──────/                  0.5 ┤──────/
     │     /                        │     /
  -1 ┤────·                    0 ┤────·
     └─────────────────         └─────────────────
       -5    0    5               -5    0    5

  출력 범위: [-1, 1]            출력 범위: [0, 1]
  0 중심 (학습에 유리)          0 중심 아님
  최대 기울기: 1                최대 기울기: 0.25
```

In [ ]:
# ============================================================
# tanh 함수와 그 도함수(derivative)를 시각화하여 직관적으로 이해하기
# 왜 시각화? → tanh의 특성을 눈으로 확인해야 기울기 소실을 이해할 수 있기 때문
# ============================================================

# -5에서 5까지 1000개의 균일한 점을 생성 (x축 값)
z = np.linspace(-5, 5, 1000)

# tanh 함수 값 계산: 출력이 [-1, 1] 범위에 있음을 확인
tanh_z = np.tanh(z)

# tanh의 도함수: 1 - tanh²(z) — 역전파(backpropagation)에서 이 값이 기울기에 곱해짐
# 왜 중요? → 이 값이 1보다 작으면 기울기가 점점 줄어듦 (기울기 소실의 원인)
dtanh_z = 1 - np.tanh(z) ** 2

# --- 그래프 그리기 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1행 2열 서브플롯 생성

# 왼쪽: tanh 함수 자체
axes[0].plot(z, tanh_z, 'b-', linewidth=2, label=r'$\tanh(z)$')  # 파란 실선
axes[0].axhline(y=0, color='k', linewidth=0.5)   # y=0 기준선 (검은 가는 선)
axes[0].axhline(y=1, color='r', linewidth=0.5, linestyle='--')   # y=1 상한선
axes[0].axhline(y=-1, color='r', linewidth=0.5, linestyle='--')  # y=-1 하한선
axes[0].set_title('tanh 활성화 함수', fontsize=14)
axes[0].set_xlabel('z (입력값)')
axes[0].set_ylabel('tanh(z) (출력값)')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)  # 격자 표시 (투명도 30%)

# 오른쪽: tanh의 도함수 — 역전파에서 핵심적인 역할
axes[1].plot(z, dtanh_z, 'r-', linewidth=2, label=r"$\tanh'(z) = 1 - \tanh^2(z)$")
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axhline(y=1, color='gray', linewidth=0.5, linestyle='--')  # 최대값 1
axes[1].set_title('tanh 도함수 (역전파에서 사용)', fontsize=14)
axes[1].set_xlabel('z (입력값)')
axes[1].set_ylabel("tanh'(z) (기울기)")
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

# |z|가 클수록 도함수가 0에 가까워짐 → 기울기 소실의 원인!
axes[1].annotate('z=0일 때 최대값 1', xy=(0, 1), xytext=(2, 0.8),
                 arrowprops=dict(arrowstyle='->', color='black'),
                 fontsize=11, color='black')
axes[1].annotate('|z|가 크면 → 0에 수렴\n(기울기 소실!)', xy=(3, 0.01), xytext=(2, 0.4),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=11, color='red')

plt.tight_layout()  # 서브플롯 간 여백 자동 조정
plt.show()

print("핵심 관찰:")
print("1. tanh 함수: 어떤 값이든 -1~1 사이로 압축합니다.")
print("2. tanh 도함수: 입력이 0에서 멀어질수록 기울기가 0에 가까워집니다.")
print("3. 결론: 시점이 길어지면 이 작은 기울기가 반복 곱해져서 기울기가 소실됩니다.")

## 1.3 순전파(Forward Pass, 앞으로 전파) 전체 과정 Step-by-Step

시퀀스 길이가 $T$인 입력 $[x_0, x_1, \ldots, x_{T-1}]$이 주어졌을 때, RNN의 순전파는 다음과 같이 진행됩니다:

> **순전파를 쉽게 말하면**: 입력 데이터를 왼쪽에서 오른쪽으로 흘려보내며 각 시점의 출력을 계산하는 과정입니다. 마치 컨베이어 벨트 위에서 재료(입력)가 차례로 들어오고, 각 단계에서 가공(계산)되어 제품(출력)이 나오는 것과 같습니다.

### Step 0: 초기 은닉 상태 설정

$$
h_{-1} = \mathbf{0} \quad \text{(영벡터, 차원: } (h, 1)\text{)}
$$

> 시퀀스 시작 전에는 "기억"이 없으므로 0으로 초기화합니다. 마치 새 학기 첫날, 아직 아무것도 배우지 않은 상태와 같습니다.

### Step 1 ~ T: 각 시점에서 반복

**시점 $t = 0$일 때:**

$$
\begin{align}
a_0 &= W_{xh} \cdot x_0 + W_{hh} \cdot h_{-1} + b_h & \text{(선형 변환: 아직 활성화 함수 적용 전)} \\
h_0 &= \tanh(a_0) & \text{(비선형 활성화: 값을 [-1,1]로 압축)} \\
y_0 &= W_{hy} \cdot h_0 + b_y & \text{(출력 계산)}
\end{align}
$$

**시점 $t = 1$일 때:**

$$
\begin{align}
a_1 &= W_{xh} \cdot x_1 + W_{hh} \cdot h_0 + b_h & \text{(h₀가 전달됨 — 이전 기억이 섞임!)} \\
h_1 &= \tanh(a_1) \\
y_1 &= W_{hy} \cdot h_1 + b_y
\end{align}
$$

**일반화 (시점 $t$):**

$$
\begin{align}
a_t &= W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h \\
h_t &= \tanh(a_t) \\
y_t &= W_{hy} \cdot h_t + b_y
\end{align}
$$

> **핵심 관찰**: 모든 시점에서 $W_{xh}, W_{hh}, W_{hy}, b_h, b_y$는 **동일한 값을 공유**합니다 (파라미터 공유, Parameter Sharing). 이것이 RNN이 가변 길이 시퀀스를 처리할 수 있는 이유입니다. 마치 같은 선생님이 매 수업마다 학생(입력)에게 같은 방식으로 가르치는 것과 같습니다.

### 순전파 흐름도

```
입력:  x₀         x₁         x₂         x₃
       ↓          ↓          ↓          ↓
      ┌──┐  h₀  ┌──┐  h₁  ┌──┐  h₂  ┌──┐
h₋₁→ │셀│─────→│셀│─────→│셀│─────→│셀│──→ h₃
      └──┘       └──┘       └──┘       └──┘
       ↓          ↓          ↓          ↓
출력:  y₀         y₁         y₂         y₃

각 셀 내부:
┌─────────────────────────────────┐
│  a_t = W_xh @ x_t              │
│      + W_hh @ h_{t-1}          │
│      + b_h                      │
│  h_t = tanh(a_t)    ← 새 기억  │
│  y_t = W_hy @ h_t + b_y ← 출력 │
└─────────────────────────────────┘
```

### 순전파 의사코드(pseudocode)

```python
h = zeros(hidden_size)           # 초기 은닉 상태 (기억 = 0)
outputs = []                     # 각 시점의 출력을 저장

for t in range(T):               # T개의 시점을 순서대로
    a = Wxh @ x[t] + Whh @ h + bh   # 선형 변환
    h = tanh(a)                      # 비선형 활성화 → 새로운 은닉 상태
    y = Why @ h + by                 # 출력 계산
    outputs.append(y)
```

## 1.4 역전파: BPTT (Backpropagation Through Time, 시간 역전파)

RNN의 역전파는 **시간을 거슬러 올라가며** 기울기(gradient)를 전파합니다. 이를 **BPTT(시간을 통한 역전파)**라고 합니다.

> **BPTT를 쉽게 말하면**: 순전파가 "왼쪽→오른쪽"이었다면, 역전파는 "오른쪽→왼쪽"으로 되돌아가며 "어떤 가중치를 얼마나 바꿔야 하는지" 계산하는 과정입니다. 마치 시험에서 틀린 문제를 거꾸로 되짚어가며 어디서 실수했는지 찾는 것과 같습니다.

### 손실 함수(Loss Function) 정의

전체 시퀀스에 대한 손실은 각 시점의 손실의 합입니다:

$$
L = \sum_{t=0}^{T-1} L_t
$$

예를 들어 MSE(평균 제곱 오차, Mean Squared Error)를 사용한다면:

$$
L_t = \frac{1}{2} \|y_t - \hat{y}_t\|^2
$$

여기서 $\hat{y}_t$는 시점 $t$의 정답(target)입니다.

> 쉽게 말하면, 손실 함수는 "RNN의 예측이 정답에서 얼마나 벗어났는지"를 숫자로 나타낸 것입니다. 이 값이 작을수록 RNN이 잘 예측한 것입니다.

### 출력층 기울기 (쉬운 부분)

수식 2($y_t = W_{hy} h_t + b_y$)에 대한 기울기는 일반 신경망과 동일합니다:

$$
\frac{\partial L_t}{\partial y_t} = y_t - \hat{y}_t \quad \text{(MSE의 경우)}
$$

$$
\frac{\partial L_t}{\partial W_{hy}} = \frac{\partial L_t}{\partial y_t} \cdot h_t^T
$$

$$
\frac{\partial L_t}{\partial b_y} = \frac{\partial L_t}{\partial y_t}
$$

### 은닉층 기울기 (BPTT의 핵심 — 어려운 부분)

$h_t$에 대한 기울기를 구해봅시다. $h_t$는 **두 가지 경로**로 손실에 영향을 미칩니다:

```
경로 1 (직접):    h_t ──→ y_t ──→ L_t        (현재 시점의 출력)
경로 2 (순환):    h_t ──→ h_{t+1} ──→ ... ──→ L_{t+1}, L_{t+2}, ...  (미래 시점으로 전달)
```

따라서:

$$
\frac{\partial L}{\partial h_t} = \underbrace{\frac{\partial L_t}{\partial h_t}}_{\text{현재 시점}} + \underbrace{\frac{\partial L}{\partial h_{t+1}} \cdot \frac{\partial h_{t+1}}{\partial h_t}}_{\text{미래에서 전파}}
$$

> 쉽게 말하면, $h_t$의 기울기는 "현재 시점에서의 오차 + 미래 시점에서 되돌아온 오차"를 합한 것입니다.

여기서 핵심 항은:

$$
\frac{\partial h_{t+1}}{\partial h_t} = \text{diag}(1 - h_{t+1}^2) \cdot W_{hh}
$$

> **해석**: $\text{diag}(1 - h_{t+1}^2)$는 tanh의 도함수를 대각행렬(diagonal matrix)로 만든 것이고, $W_{hh}$는 은닉→은닉 가중치입니다.

### $\frac{\partial L}{\partial W_{hh}}$의 전개 — BPTT의 핵심

$W_{hh}$는 **모든 시점에서 공유**되므로, 기울기는 모든 시점의 기여를 합산해야 합니다:

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=0}^{T-1} \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial a_t} \cdot \frac{\partial a_t}{\partial W_{hh}}
$$

여기서:
- $\frac{\partial h_t}{\partial a_t} = \text{diag}(1 - h_t^2)$ — tanh의 도함수
- $\frac{\partial a_t}{\partial W_{hh}} = h_{t-1}^T$ — $a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h$에서 $W_{hh}$로 미분

**$\frac{\partial L}{\partial h_t}$를 재귀적으로 풀면:**

시점 $t$에서의 기울기가 과거 시점 $k$까지 전파될 때:

$$
\frac{\partial L_t}{\partial h_k} = \frac{\partial L_t}{\partial h_t} \prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}} = \frac{\partial L_t}{\partial h_t} \prod_{j=k+1}^{t} \text{diag}(1 - h_j^2) \cdot W_{hh}
$$

> 쉽게 말하면, 기울기가 시점 $t$에서 시점 $k$까지 전달될 때, 그 사이의 모든 시점에서 "tanh 도함수 $\times$ $W_{hh}$"가 **연쇄적으로 곱해집니다**. 시점 차이($t - k$)가 클수록 곱하는 횟수가 많아집니다.

이 **연쇄 곱(chain of products)**이 BPTT의 핵심이자, 기울기 소실/폭발의 원인입니다.

### 기울기 소실/폭발 문제 — 수학적 설명

위 식에서 $\prod_{j=k+1}^{t} \text{diag}(1 - h_j^2) \cdot W_{hh}$ 부분을 살펴봅시다.

**기울기 소실 (Vanishing Gradient, 기울기가 사라짐):**

$$
\left\| \prod_{j=k+1}^{t} \text{diag}(1 - h_j^2) \cdot W_{hh} \right\| \leq \prod_{j=k+1}^{t} \underbrace{\|1 - h_j^2\|}_{\leq 1} \cdot \underbrace{\|W_{hh}\|}_{\text{보통 } < 1}
$$

- $\tanh'$의 최대값이 1이고, $W_{hh}$의 특이값(singular value)이 1보다 작으면
- $(t-k)$번 곱하면 기울기가 **지수적으로 0에 수렴** → 먼 과거의 정보를 학습할 수 없음

> **비유**: 0.9를 10번 곱하면 $0.9^{10} \approx 0.35$, 50번 곱하면 $0.9^{50} \approx 0.005$. 시점이 멀어질수록 기울기가 거의 0이 됩니다.

**기울기 폭발 (Exploding Gradient, 기울기가 폭주함):**

- 반대로 $W_{hh}$의 특이값이 1보다 크면
- $(t-k)$번 곱하면 기울기가 **지수적으로 발산** → 학습이 불안정

$$
\text{기울기 크기} \approx \lambda_{\max}(W_{hh})^{t-k}
\begin{cases}
\to 0 & \text{if } \lambda_{\max} < 1 \quad \text{(소실)} \\
\to \infty & \text{if } \lambda_{\max} > 1 \quad \text{(폭발)}
\end{cases}
$$

> **비유**: 1.1을 50번 곱하면 $1.1^{50} \approx 117$. 작은 차이가 반복되면 결과가 폭발적으로 커집니다.

> **결론**: Vanilla RNN은 **장기 의존성(Long-term Dependency, 먼 과거 정보에 의존하는 패턴)**을 학습하기 어렵습니다. 이것이 LSTM/GRU가 등장한 이유입니다.

## 1.5 RNN 구조의 변형: Many-to-One, Many-to-Many, One-to-Many

RNN은 입력/출력의 길이 조합에 따라 다양한 구조로 사용됩니다:

### (1) Many-to-One (다대일)

```
x₀   x₁   x₂   x₃
 ↓    ↓    ↓    ↓
[h₀]→[h₁]→[h₂]→[h₃]
                  ↓
                  y    ← 마지막 은닉 상태만 사용
```

- **용도**: 감정 분류, 스팸 탐지
- **동작**: 시퀀스 전체를 읽은 후, **마지막 은닉 상태** $h_{T-1}$만으로 출력을 생성
- **예시**: "이 영화 정말 재미있다" → 긍정(1)

### (2) Many-to-Many (다대다) — 같은 길이

```
x₀   x₁   x₂   x₃
 ↓    ↓    ↓    ↓
[h₀]→[h₁]→[h₂]→[h₃]
 ↓    ↓    ↓    ↓
 y₀   y₁   y₂   y₃
```

- **용도**: 품사 태깅(POS tagging), 개체명 인식(NER)
- **동작**: 매 시점마다 입력을 받고 출력을 생성
- **예시**: "나는/대명사 밥을/명사 먹었다/동사"

### (3) Many-to-Many (다대다) — 다른 길이 (Seq2Seq, Encoder-Decoder)

```
인코더:                      디코더:
x₀   x₁   x₂              <s>   y₀   y₁
 ↓    ↓    ↓                ↓     ↓    ↓
[h₀]→[h₁]→[h₂] ──context──→[s₀]→[s₁]→[s₂]
                             ↓     ↓    ↓
                             y₀    y₁   y₂
```

- **용도**: 기계 번역, 요약
- **동작**: 인코더가 입력을 요약 → 디코더가 출력을 생성
- **예시**: "I love you" → "나는 너를 사랑해"

### (4) One-to-Many (일대다)

```
x₀
 ↓
[h₀]→[h₁]→[h₂]→[h₃]
 ↓    ↓    ↓    ↓
 y₀   y₁   y₂   y₃
```

- **용도**: 이미지 캡셔닝, 음악 생성
- **동작**: 하나의 입력(예: 이미지 특징)으로부터 시퀀스를 생성
- **예시**: [사진] → "해변에서 뛰어노는 강아지"

### (5) One-to-One (일대일)

이건 그냥 일반 신경망(MLP)과 같으므로 RNN을 쓸 이유가 없습니다.

> **이 노트북에서는 Part 2에서 Many-to-Many, Part 3에서 Many-to-One을 직접 구현합니다.**

---

# Part 2: RNN 밑바닥 구현 (NumPy Only)

---

## 2.1 RNNCell 클래스 구현

이제 위에서 배운 수식을 **한 줄씩** NumPy 코드로 옮깁니다. 모든 줄에 해당 수식을 주석으로 표시합니다.

> **밑바닥 구현을 하는 이유**: PyTorch 같은 프레임워크가 내부에서 무엇을 하는지 이해해야, 문제가 생겼을 때 원인을 찾을 수 있습니다. 마치 자동차 운전만 할 줄 아는 것보다, 엔진 구조를 아는 사람이 고장 원인을 더 빨리 파악하는 것과 같습니다.

### 구현할 메서드:
1. `__init__`: 가중치 초기화 (수식의 $W_{xh}, W_{hh}, W_{hy}, b_h, b_y$ 생성)
2. `forward`: 순전파 (수식 1, 2 계산)
3. `backward`: 역전파 (BPTT로 기울기 계산 + 가중치 업데이트)

### 클래스 구조 개요

```
RNNCell
├── __init__(input_size, hidden_size, output_size)
│   ├── W_xh: (h, d)  ← 입력→은닉
│   ├── W_hh: (h, h)  ← 은닉→은닉
│   ├── W_hy: (o, h)  ← 은닉→출력
│   ├── b_h:  (h, 1)  ← 은닉 편향
│   └── b_y:  (o, 1)  ← 출력 편향
│
├── forward(inputs, h_prev)
│   └── 시점 t=0~T-1: 수식 1, 2 반복 적용
│
└── backward(d_outputs, learning_rate)
    └── 시점 t=T-1~0: BPTT로 기울기 역전파
```

In [ ]:
class RNNCell:
    """
    Vanilla RNN 셀을 NumPy로 밑바닥부터 구현한 클래스.

    핵심 수식:
        h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)   ... (수식 1)
        y_t = W_hy @ h_t + b_y                             ... (수식 2)

    Parameters:
        input_size (int): 입력 벡터의 차원 d
        hidden_size (int): 은닉 상태의 차원 h
        output_size (int): 출력 벡터의 차원 o
    """

    def __init__(self, input_size: int, hidden_size: int, output_size: int):
        """
        가중치와 편향을 초기화하는 메서드.
        Xavier 초기화를 사용하여 학습 초기에 기울기가 적절한 크기를 유지하도록 합니다.
        """
        # --- 차원 저장 ---
        self.input_size = input_size    # d: 입력 벡터 차원
        self.hidden_size = hidden_size  # h: 은닉 상태 차원
        self.output_size = output_size  # o: 출력 벡터 차원

        # --- 가중치 초기화 (Xavier/Glorot) ---
        # W_xh: 입력 → 은닉 변환 행렬, 차원 (h, d)
        # 수식 1에서 W_xh @ x_t 부분에 사용
        # Xavier 초기화: 표준편차 = sqrt(2 / (fan_in + fan_out))
        self.W_xh = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / (input_size + hidden_size))

        # W_hh: 은닉 → 은닉 변환 행렬, 차원 (h, h)
        # 수식 1에서 W_hh @ h_{t-1} 부분에 사용
        # 이 행렬이 매 시점마다 곱해지므로 기울기 소실/폭발의 핵심 원인!
        self.W_hh = np.random.randn(hidden_size, hidden_size) * np.sqrt(2.0 / (hidden_size + hidden_size))

        # W_hy: 은닉 → 출력 변환 행렬, 차원 (o, h)
        # 수식 2에서 W_hy @ h_t 부분에 사용
        self.W_hy = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / (hidden_size + output_size))

        # b_h: 은닉 상태 편향, 차원 (h, 1)
        # 수식 1에서 + b_h 부분
        self.b_h = np.zeros((hidden_size, 1))

        # b_y: 출력 편향, 차원 (o, 1)
        # 수식 2에서 + b_y 부분
        self.b_y = np.zeros((output_size, 1))

        # --- 기울기 저장용 변수 (backward에서 채워짐) ---
        # 각 가중치/편향에 대한 기울기를 누적할 공간
        self.dW_xh = np.zeros_like(self.W_xh)  # dL/dW_xh 누적
        self.dW_hh = np.zeros_like(self.W_hh)  # dL/dW_hh 누적
        self.dW_hy = np.zeros_like(self.W_hy)  # dL/dW_hy 누적
        self.db_h = np.zeros_like(self.b_h)    # dL/db_h 누적
        self.db_y = np.zeros_like(self.b_y)    # dL/db_y 누적

    def forward(self, inputs, h_prev=None):
        """
        순전파: 시퀀스 전체를 시점별로 처리합니다.

        Args:
            inputs: 입력 시퀀스, shape = (T, d, 1)
                    T = 시퀀스 길이, d = 입력 차원
            h_prev: 초기 은닉 상태, shape = (h, 1)
                    None이면 영벡터로 초기화

        Returns:
            outputs: 각 시점의 출력 리스트, 각 원소 shape = (o, 1)
            hiddens: 각 시점의 은닉 상태 리스트 (역전파에서 필요)
        """
        T = len(inputs)  # 시퀀스 길이 (총 몇 개의 시점이 있는지)

        # 초기 은닉 상태가 없으면 영벡터로 시작
        # h_{-1} = 0 (시작 전에는 기억이 없음)
        if h_prev is None:
            h_prev = np.zeros((self.hidden_size, 1))

        # --- 순전파 중간값 저장 (역전파에서 필요) ---
        self.inputs = inputs            # 입력 시퀀스 저장
        self.hiddens = [h_prev]         # h_{-1}부터 저장 (역전파에서 h_{t-1} 참조용)
        self.a_values = []              # tanh 적용 전의 값 a_t 저장
        outputs = []                    # 각 시점의 출력 y_t 저장

        h = h_prev  # 현재 은닉 상태를 h로 추적

        for t in range(T):
            x_t = inputs[t]  # 시점 t의 입력 벡터, shape = (d, 1)

            # ====================================
            # 수식 1: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
            # ====================================

            # Step 1a: W_xh @ x_t → 입력을 은닉 공간으로 변환
            # 차원: (h, d) @ (d, 1) = (h, 1)
            xh = self.W_xh @ x_t

            # Step 1b: W_hh @ h_{t-1} → 이전 기억을 현재로 전달
            # 차원: (h, h) @ (h, 1) = (h, 1)
            hh = self.W_hh @ h

            # Step 1c: 두 변환을 더하고 편향 추가 → 활성화 함수 적용 전 값 a_t
            # a_t = W_xh @ x_t + W_hh @ h_{t-1} + b_h
            a_t = xh + hh + self.b_h

            # Step 1d: tanh 활성화 함수 적용 → 새로운 은닉 상태 h_t
            # tanh는 값을 [-1, 1] 범위로 압축하여 발산을 방지
            h = np.tanh(a_t)

            # ====================================
            # 수식 2: y_t = W_hy @ h_t + b_y
            # ====================================

            # 은닉 상태를 출력 공간으로 변환
            # 차원: (o, h) @ (h, 1) + (o, 1) = (o, 1)
            y_t = self.W_hy @ h + self.b_y

            # 중간값 저장 (역전파에서 사용)
            self.a_values.append(a_t)   # tanh 적용 전 값
            self.hiddens.append(h)      # 시점 t의 은닉 상태
            outputs.append(y_t)         # 시점 t의 출력

        return outputs, self.hiddens

    def backward(self, d_outputs, learning_rate=0.01):
        """
        BPTT (Backpropagation Through Time): 시간을 거슬러 올라가며 기울기를 계산합니다.

        Args:
            d_outputs: 각 시점의 출력에 대한 손실의 기울기 리스트
                       d_outputs[t] = dL/dy_t, shape = (o, 1)
            learning_rate: 학습률 (가중치 업데이트 크기)

        수식 유도 (Part 1.4 참조):
            dL/dW_hy = sum_t (dL/dy_t @ h_t^T)
            dL/db_y  = sum_t (dL/dy_t)
            dL/dh_t  = W_hy^T @ dL/dy_t + dh_next (미래에서 전파된 기울기)
            dL/da_t  = dL/dh_t * (1 - h_t^2)     ← tanh의 도함수!
            dL/dW_xh = sum_t (dL/da_t @ x_t^T)
            dL/dW_hh = sum_t (dL/da_t @ h_{t-1}^T)
            dL/db_h  = sum_t (dL/da_t)
        """
        T = len(d_outputs)  # 시퀀스 길이

        # 기울기 초기화 (이전 backward 호출의 값이 남아있을 수 있으므로)
        self.dW_xh = np.zeros_like(self.W_xh)  # (h, d)
        self.dW_hh = np.zeros_like(self.W_hh)  # (h, h)
        self.dW_hy = np.zeros_like(self.W_hy)  # (o, h)
        self.db_h = np.zeros_like(self.b_h)    # (h, 1)
        self.db_y = np.zeros_like(self.b_y)    # (o, 1)

        # 미래 시점에서 전파되는 은닉 상태 기울기
        # 마지막 시점 이후에는 미래가 없으므로 0으로 시작
        dh_next = np.zeros((self.hidden_size, 1))

        # --- 시점별 기울기 크기를 기록 (기울기 소실 시각화용) ---
        self.grad_norms = []  # 각 시점에서의 기울기 크기(norm)를 저장

        # ====================================
        # 시간을 역순으로 거슬러 올라감 (BPTT의 핵심!)
        # t = T-1, T-2, ..., 1, 0 순서
        # ====================================
        for t in reversed(range(T)):
            # --- 시점 t에서 필요한 값들 꺼내기 ---
            dy_t = d_outputs[t]            # dL/dy_t: 시점 t에서 출력의 기울기
            h_t = self.hiddens[t + 1]      # h_t: 시점 t의 은닉 상태 (hiddens[0]이 h_{-1}이므로 +1)
            h_prev = self.hiddens[t]       # h_{t-1}: 이전 시점의 은닉 상태
            x_t = self.inputs[t]           # x_t: 시점 t의 입력

            # ====================================
            # Step 1: 출력층 기울기 (수식 2의 역전파)
            # y_t = W_hy @ h_t + b_y
            # ====================================

            # dL/dW_hy += dL/dy_t @ h_t^T
            # 차원: (o, 1) @ (1, h) = (o, h) ← W_hy와 같은 차원!
            self.dW_hy += dy_t @ h_t.T

            # dL/db_y += dL/dy_t
            # 편향의 기울기는 출력 기울기 그 자체
            self.db_y += dy_t

            # ====================================
            # Step 2: 은닉 상태 기울기 (BPTT 핵심)
            # h_t는 두 경로로 손실에 기여:
            #   (1) 현재 출력: h_t → y_t → L_t
            #   (2) 미래 전파: h_t → h_{t+1} → ... → L_{t+1}, ...
            # ====================================

            # dL/dh_t = W_hy^T @ dL/dy_t + dh_next
            # 첫 번째 항: 현재 시점 출력을 통한 기울기
            # 두 번째 항: 미래 시점에서 전파된 기울기
            # 차원: (h, o) @ (o, 1) + (h, 1) = (h, 1)
            dh = self.W_hy.T @ dy_t + dh_next

            # ====================================
            # Step 3: tanh의 역전파
            # h_t = tanh(a_t) 이므로
            # dL/da_t = dL/dh_t * tanh'(a_t) = dL/dh_t * (1 - h_t^2)
            # ====================================

            # (1 - h_t^2)는 tanh의 도함수 — Part 1.2에서 유도한 것!
            # 원소별 곱셈 (*): 각 은닉 유닛별로 독립적으로 적용
            da = dh * (1 - h_t ** 2)

            # 기울기 크기 기록 (시각화용)
            self.grad_norms.append(np.linalg.norm(da))

            # ====================================
            # Step 4: 가중치/편향 기울기 계산 (수식 1의 역전파)
            # a_t = W_xh @ x_t + W_hh @ h_{t-1} + b_h
            # ====================================

            # dL/dW_xh += dL/da_t @ x_t^T
            # a_t에서 W_xh로의 기울기: x_t가 곱해져 있으므로 x_t^T를 곱함
            # 차원: (h, 1) @ (1, d) = (h, d) ← W_xh와 같은 차원!
            self.dW_xh += da @ x_t.T

            # dL/dW_hh += dL/da_t @ h_{t-1}^T
            # a_t에서 W_hh로의 기울기: h_{t-1}이 곱해져 있으므로 h_{t-1}^T를 곱함
            # 차원: (h, 1) @ (1, h) = (h, h) ← W_hh와 같은 차원!
            self.dW_hh += da @ h_prev.T

            # dL/db_h += dL/da_t
            # 편향은 그대로 더해지므로 기울기도 그대로
            self.db_h += da

            # ====================================
            # Step 5: 이전 시점으로 기울기 전파 (BPTT의 재귀)
            # dL/dh_{t-1} = W_hh^T @ dL/da_t
            # 이 값이 다음 반복(t-1 시점)에서 dh_next로 사용됨
            # ====================================
            dh_next = self.W_hh.T @ da

        # 기울기 크기를 시간 순서로 정렬 (reversed로 계산했으므로 뒤집기)
        self.grad_norms = list(reversed(self.grad_norms))

        # ====================================
        # Step 6: 기울기 클리핑 (Gradient Clipping)
        # 기울기 폭발을 방지하기 위해 기울기의 크기를 제한
        # ====================================
        clip_value = 5.0  # 기울기의 최대 허용 크기
        for param_grad in [self.dW_xh, self.dW_hh, self.dW_hy, self.db_h, self.db_y]:
            # 각 기울기를 [-clip_value, clip_value] 범위로 자름
            np.clip(param_grad, -clip_value, clip_value, out=param_grad)

        # ====================================
        # Step 7: 가중치 업데이트 (경사 하강법)
        # θ ← θ - lr * dL/dθ
        # ====================================
        self.W_xh -= learning_rate * self.dW_xh  # 입력→은닉 가중치 업데이트
        self.W_hh -= learning_rate * self.dW_hh  # 은닉→은닉 가중치 업데이트
        self.W_hy -= learning_rate * self.dW_hy  # 은닉→출력 가중치 업데이트
        self.b_h -= learning_rate * self.db_h    # 은닉 편향 업데이트
        self.b_y -= learning_rate * self.db_y    # 출력 편향 업데이트

        return self.grad_norms  # 기울기 시각화를 위해 반환

print("RNNCell 클래스가 정의되었습니다!")
print(f"총 메서드: __init__, forward, backward")
print(f"핵심 가중치: W_xh(입력→은닉), W_hh(은닉→은닉), W_hy(은닉→출력)")

## 2.2 작은 예제로 순전파 직접 계산하여 값 확인

아주 작은 차원으로 RNN의 순전파를 직접 손으로 따라가 봅시다.

- 입력 차원 $d = 2$ (2차원 벡터)
- 은닉 차원 $h = 3$ (3개의 은닉 유닛)
- 출력 차원 $o = 1$ (스칼라 출력)
- 시퀀스 길이 $T = 4$ (4개 시점)

### 예제의 데이터 흐름

```
x₀ (2,1)  x₁ (2,1)  x₂ (2,1)  x₃ (2,1)     ← 입력: 2차원 벡터 4개
    ↓          ↓          ↓          ↓
  ┌───┐ h₀  ┌───┐ h₁  ┌───┐ h₂  ┌───┐
→ │3×2│────→│3×3│────→│3×3│────→│3×3│──→ h₃ (3,1)
  │셀 │     │셀 │     │셀 │     │셀 │       ← 은닉: 3차원 벡터
  └───┘     └───┘     └───┘     └───┘
    ↓          ↓          ↓          ↓
  y₀ (1,1)  y₁ (1,1)  y₂ (1,1)  y₃ (1,1)   ← 출력: 스칼라 1개
```

> 이 예제에서는 랜덤 입력을 사용하지만, 실제로는 단어 임베딩(word embedding)이나 센서 측정값 등이 들어갑니다.

In [ ]:
# ============================================================
# 2.2 작은 예제로 순전파를 직접 계산
# ============================================================

# --- 하이퍼파라미터 설정 ---
d = 2   # 입력 차원: 각 시점에서 2차원 벡터가 들어옴
h = 3   # 은닉 차원: RNN이 3개의 은닉 유닛으로 정보를 저장
o = 1   # 출력 차원: 각 시점에서 스칼라 1개를 출력
T = 4   # 시퀀스 길이: 4개의 시점

# --- 난수 시드 고정 ---
np.random.seed(42)  # 재현 가능한 결과를 위해

# --- RNN 셀 생성 ---
rnn = RNNCell(input_size=d, hidden_size=h, output_size=o)

# --- 입력 데이터 생성 ---
# 4개 시점, 각각 2차원 벡터 (shape: (T, d, 1))
# 실제로는 단어 임베딩이나 센서 데이터가 들어가지만, 여기선 랜덤 값 사용
inputs = [np.random.randn(d, 1) for _ in range(T)]

print("=" * 60)
print("입력 시퀀스 (4개 시점, 각 2차원):")
print("=" * 60)
for t, x in enumerate(inputs):
    # 각 시점의 입력 벡터를 출력
    print(f"  x_{t} = {x.flatten()}  (shape: {x.shape})")

# --- 가중치 확인 ---
print(f"\n가중치 차원:")
print(f"  W_xh: {rnn.W_xh.shape} (입력→은닉: {d}차원 → {h}차원)")
print(f"  W_hh: {rnn.W_hh.shape} (은닉→은닉: {h}차원 → {h}차원)")
print(f"  W_hy: {rnn.W_hy.shape} (은닉→출력: {h}차원 → {o}차원)")

In [ ]:
# ============================================================
# 순전파 실행 및 각 시점별 상세 출력
# ============================================================

# 순전파 실행
outputs, hiddens = rnn.forward(inputs)

print("=" * 60)
print("순전파 결과 (각 시점별 상세)")
print("=" * 60)

for t in range(T):
    print(f"\n--- 시점 t={t} ---")

    # 입력 벡터
    print(f"  입력 x_{t}: {inputs[t].flatten()}")

    # 이전 은닉 상태 (t=0이면 영벡터)
    print(f"  이전 은닉 h_{t-1 if t > 0 else '초기'}: {hiddens[t].flatten()}")

    # 수식 1의 각 단계를 재현하여 보여줌
    xh = rnn.W_xh @ inputs[t]           # W_xh @ x_t
    hh = rnn.W_hh @ hiddens[t]          # W_hh @ h_{t-1}
    a_t = xh + hh + rnn.b_h             # 활성화 함수 적용 전
    h_t = np.tanh(a_t)                   # tanh 적용 후

    print(f"  W_xh @ x_{t} = {xh.flatten()}")       # 입력 변환 결과
    print(f"  W_hh @ h_{t-1 if t > 0 else '초기'} = {hh.flatten()}")  # 은닉 변환 결과
    print(f"  a_{t} (tanh 전) = {a_t.flatten()}")    # 활성화 전 값
    print(f"  h_{t} (tanh 후) = {h_t.flatten()}")    # 새 은닉 상태
    print(f"  y_{t} (출력)    = {outputs[t].flatten()}")  # 출력값

    # 우리 RNN 클래스의 결과와 일치하는지 검증
    assert np.allclose(h_t, hiddens[t + 1]), "은닉 상태가 일치하지 않음!"

print("\n" + "=" * 60)
print("모든 시점에서 수동 계산과 RNNCell.forward()의 결과가 일치합니다!")
print("=" * 60)

## 2.3 역전파(BPTT) 직접 계산 및 기울기 검증

역전파가 정확한지 확인하기 위해 **수치 기울기(numerical gradient)**와 비교합니다.

### 수치 기울기란?

미분의 정의를 직접 계산하는 것입니다:

$$
\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}
$$

> 쉽게 말하면, 가중치를 아주 조금($\epsilon$) 올렸을 때의 손실과, 아주 조금 내렸을 때의 손실 차이를 비교하여 기울기를 직접 측정하는 것입니다. 마치 온도계 없이 물을 손으로 만져보고 "약간 더 뜨겁다/차갑다"고 판단하는 것과 비슷합니다.

이 값이 우리 `backward()`가 계산한 기울기와 거의 같으면 구현이 올바른 것입니다.

### 검증 방법 요약

| 단계 | 방법 | 비교 대상 |
|------|------|-----------|
| 1 | `backward()`로 해석적 기울기 계산 | $\frac{\partial L}{\partial W_{hh}}$ (수식으로 계산) |
| 2 | $\epsilon$-perturbation으로 수치 기울기 계산 | $\frac{L(\theta+\epsilon) - L(\theta-\epsilon)}{2\epsilon}$ |
| 3 | 두 값의 상대 오차 확인 | $< 10^{-5}$이면 통과 |

In [ ]:
# ============================================================
# 2.3 역전파 검증: 수치 기울기(numerical gradient)와 비교
# ============================================================

np.random.seed(42)  # 재현성을 위해 시드 고정

# --- 작은 RNN 생성 ---
d_test, h_test, o_test = 2, 3, 1  # 작은 차원으로 테스트
T_test = 4                         # 시퀀스 길이 4
rnn_test = RNNCell(d_test, h_test, o_test)

# --- 입력 데이터 및 타겟 생성 ---
inputs_test = [np.random.randn(d_test, 1) for _ in range(T_test)]
targets_test = [np.random.randn(o_test, 1) for _ in range(T_test)]

# --- 순전파 실행 ---
outputs_test, _ = rnn_test.forward(inputs_test)

# --- 손실 계산 (MSE: Mean Squared Error) ---
# L = (1/2) * sum_t ||y_t - target_t||^2
loss = 0
d_outputs_test = []  # dL/dy_t를 저장할 리스트
for t in range(T_test):
    diff = outputs_test[t] - targets_test[t]  # y_t - target_t
    loss += 0.5 * np.sum(diff ** 2)           # (1/2)||y_t - target_t||^2
    d_outputs_test.append(diff)                # dL/dy_t = y_t - target_t (MSE의 기울기)

print(f"현재 손실(MSE): {loss:.6f}")

# --- 역전파 실행 (가중치 업데이트 없이 기울기만 계산) ---
# 학습률을 0으로 설정하면 기울기만 계산하고 가중치는 변경하지 않음
lr_zero = 0.0

# 가중치 복사 (나중에 수치 기울기 계산 시 원본 가중치 필요)
W_xh_copy = rnn_test.W_xh.copy()
W_hh_copy = rnn_test.W_hh.copy()
W_hy_copy = rnn_test.W_hy.copy()
b_h_copy = rnn_test.b_h.copy()
b_y_copy = rnn_test.b_y.copy()

# 역전파로 해석적 기울기 계산
rnn_test.backward(d_outputs_test, learning_rate=lr_zero)
analytic_dW_hh = rnn_test.dW_hh.copy()  # backward에서 계산한 dL/dW_hh

print(f"\n해석적 기울기 (backward)로 계산한 dL/dW_hh:")
print(analytic_dW_hh)

# --- 수치 기울기 계산 ---
# ε만큼 가중치를 앞뒤로 흔들어서 손실 변화를 관찰
epsilon = 1e-5  # 아주 작은 값
numerical_dW_hh = np.zeros_like(rnn_test.W_hh)  # 수치 기울기 저장

for i in range(h_test):       # W_hh의 행 인덱스
    for j in range(h_test):   # W_hh의 열 인덱스
        # W_hh[i,j]를 +ε만큼 증가
        rnn_test.W_hh = W_hh_copy.copy()  # 원본 복원
        rnn_test.W_hh[i, j] += epsilon
        out_plus, _ = rnn_test.forward(inputs_test)  # 순전파
        loss_plus = sum(0.5 * np.sum((out_plus[t] - targets_test[t])**2) for t in range(T_test))

        # W_hh[i,j]를 -ε만큼 감소
        rnn_test.W_hh = W_hh_copy.copy()  # 원본 복원
        rnn_test.W_hh[i, j] -= epsilon
        out_minus, _ = rnn_test.forward(inputs_test)  # 순전파
        loss_minus = sum(0.5 * np.sum((out_minus[t] - targets_test[t])**2) for t in range(T_test))

        # 수치 기울기: (L(θ+ε) - L(θ-ε)) / 2ε
        numerical_dW_hh[i, j] = (loss_plus - loss_minus) / (2 * epsilon)

# 원본 가중치 복원
rnn_test.W_hh = W_hh_copy.copy()

print(f"\n수치 기울기 (numerical gradient)로 계산한 dL/dW_hh:")
print(numerical_dW_hh)

# --- 두 기울기의 차이 비교 ---
diff = np.abs(analytic_dW_hh - numerical_dW_hh)
relative_error = np.max(diff) / (np.max(np.abs(analytic_dW_hh)) + 1e-8)
print(f"\n최대 절대 오차: {np.max(diff):.2e}")
print(f"상대 오차: {relative_error:.2e}")

if relative_error < 1e-5:
    print("\n기울기 검증 통과! 역전파 구현이 올바릅니다!")
else:
    print("\n경고: 기울기 차이가 크므로 역전파 구현을 확인해주세요.")

## 2.4 사인파 예측 실습: RNN으로 시계열(time series) 학습하기

이제 실제로 RNN을 **학습**시켜봅시다! 사인파($\sin$ 함수)의 다음 값을 예측하는 문제입니다.

- **입력**: $[\sin(t), \sin(t+1), \ldots, \sin(t+T-1)]$ (T개의 연속 값)
- **타겟**: $[\sin(t+1), \sin(t+2), \ldots, \sin(t+T)]$ (한 시점씩 앞선 값)
- **구조**: Many-to-Many (매 시점마다 다음 값을 예측)

> **사인파를 선택한 이유**: 사인파는 규칙적인 패턴이 있어서 RNN이 학습하기에 적절한 난이도입니다. 마치 "봄-여름-가을-겨울"이 반복되는 패턴을 학습하는 것과 비슷합니다.

### 데이터 구성 예시

```
입력 시퀀스 (T=5):  [sin(0), sin(1), sin(2), sin(3), sin(4)]
타겟 시퀀스 (T=5):  [sin(1), sin(2), sin(3), sin(4), sin(5)]
                      ↑       ↑       ↑       ↑       ↑
                    한 칸씩 미래의 값을 예측!

시각화:
      1 ┤    *         *
        │   / \       / \
      0 ┤──/───\─────/───\──→ 시간
        │ /     \   /     \
     -1 ┤*       \ /       *
                   *
          ├─입력──┤├─예측──┤
```

In [ ]:
# ============================================================
# 2.4 사인파 예측: 데이터 생성 및 학습
# ============================================================

np.random.seed(42)  # 재현성

# --- 사인파 데이터 생성 ---
# 0부터 20π까지의 사인파를 0.1 간격으로 샘플링
time_steps = np.arange(0, 20 * np.pi, 0.1)  # 약 628개의 시점
sin_wave = np.sin(time_steps)  # 사인파 값

print(f"전체 데이터 포인트 수: {len(sin_wave)}")
print(f"처음 10개 값: {sin_wave[:10].round(3)}")

# --- 시퀀스 데이터 생성 ---
# 시퀀스 길이 T=25: 25개의 연속 값으로 다음 25개를 예측
seq_len = 25  # 한 번에 볼 시퀀스 길이

# 입력-타겟 쌍 만들기
X_sequences = []  # 입력 시퀀스들
Y_sequences = []  # 타겟 시퀀스들

for i in range(len(sin_wave) - seq_len - 1):
    # 입력: [sin(i), sin(i+1), ..., sin(i+seq_len-1)]
    # 각 값을 (1, 1) shape로 변환 (입력 차원 d=1)
    x_seq = [np.array([[sin_wave[i + t]]]) for t in range(seq_len)]
    # 타겟: [sin(i+1), sin(i+2), ..., sin(i+seq_len)]
    y_seq = [np.array([[sin_wave[i + t + 1]]]) for t in range(seq_len)]
    X_sequences.append(x_seq)
    Y_sequences.append(y_seq)

print(f"학습 시퀀스 수: {len(X_sequences)}")

# --- RNN 생성 ---
# 입력 차원 d=1 (사인파 값 1개), 은닉 차원 h=16, 출력 차원 o=1
rnn_sine = RNNCell(input_size=1, hidden_size=16, output_size=1)

# --- 학습 루프 ---
num_epochs = 100    # 전체 데이터를 100번 반복 학습
learning_rate = 0.005  # 학습률 (너무 크면 발산, 너무 작으면 학습이 느림)
loss_history = []   # 에포크별 손실 기록

# 학습 데이터 중 처음 200개 시퀀스만 사용 (빠른 실행을 위해)
num_train = 200

for epoch in range(num_epochs):
    epoch_loss = 0  # 이 에포크의 총 손실

    for seq_idx in range(num_train):
        # --- 순전파 ---
        x_seq = X_sequences[seq_idx]   # 입력 시퀀스
        y_seq = Y_sequences[seq_idx]   # 타겟 시퀀스

        # forward: 시퀀스를 RNN에 통과
        outputs, hiddens = rnn_sine.forward(x_seq)

        # --- 손실 계산 (MSE) ---
        d_outputs = []  # 각 시점의 출력 기울기
        for t in range(seq_len):
            diff = outputs[t] - y_seq[t]  # 예측 - 정답
            epoch_loss += 0.5 * np.sum(diff ** 2)  # MSE 손실
            d_outputs.append(diff)  # dL/dy_t = y_t - target_t

        # --- 역전파 + 가중치 업데이트 ---
        rnn_sine.backward(d_outputs, learning_rate=learning_rate)

    # 에포크 평균 손실 기록
    avg_loss = epoch_loss / (num_train * seq_len)  # 시퀀스당, 시점당 평균 손실
    loss_history.append(avg_loss)

    # 10 에포크마다 진행 상황 출력
    if (epoch + 1) % 10 == 0:
        print(f"에포크 {epoch+1:3d}/{num_epochs} | 평균 손실: {avg_loss:.6f}")

print("\n학습 완료!")

In [ ]:
# ============================================================
# 학습 결과 시각화: 손실 곡선 + 예측 vs 실제
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- (1) 손실 곡선 ---
axes[0].plot(loss_history, 'b-', linewidth=1.5)  # 에포크별 손실
axes[0].set_title('학습 손실 곡선 (에포크별 평균 MSE)', fontsize=14)
axes[0].set_xlabel('에포크')
axes[0].set_ylabel('평균 손실')
axes[0].set_yscale('log')  # 로그 스케일로 보면 감소 추이가 더 잘 보임
axes[0].grid(True, alpha=0.3)
axes[0].annotate(f'최종 손실: {loss_history[-1]:.6f}',
                 xy=(len(loss_history)-1, loss_history[-1]),
                 xytext=(len(loss_history)*0.5, loss_history[-1]*5),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=11, color='red')

# --- (2) 예측 vs 실제 ---
# 학습에 사용하지 않은 시퀀스로 예측
test_idx = 300  # 테스트용 시퀀스 인덱스
x_test = X_sequences[test_idx]
y_test = Y_sequences[test_idx]

# 순전파로 예측
pred_outputs, _ = rnn_sine.forward(x_test)

# 실제값과 예측값 추출
actual = [y_test[t].item() for t in range(seq_len)]       # 실제 다음 값
predicted = [pred_outputs[t].item() for t in range(seq_len)]  # RNN이 예측한 값
input_vals = [x_test[t].item() for t in range(seq_len)]   # 입력값

# 시각화
t_axis = range(seq_len)  # x축: 시점
axes[1].plot(t_axis, input_vals, 'g--', linewidth=1, label='입력 (현재값)', alpha=0.5)
axes[1].plot(t_axis, actual, 'b-o', markersize=4, linewidth=2, label='실제 (다음값)')
axes[1].plot(t_axis, predicted, 'r--s', markersize=4, linewidth=2, label='예측 (RNN)')
axes[1].set_title('사인파 예측: 실제 vs RNN 예측', fontsize=14)
axes[1].set_xlabel('시점 (t)')
axes[1].set_ylabel('값')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("파란선(실제)과 빨간선(예측)이 가까울수록 RNN이 잘 학습한 것입니다!")

## 2.5 기울기 소실(Vanishing Gradient) 시각화

BPTT에서 **시점이 멀어질수록 기울기가 얼마나 줄어드는지** 실제로 측정하여 시각화합니다.

이것이 바로 Part 1.4에서 수학적으로 유도한 **기울기 소실** 문제입니다.

> **실험 설계**: $W_{hh}$의 크기(스케일)를 0.5, 0.9, 1.0, 1.5로 바꿔가며 기울기가 어떻게 변하는지 관찰합니다.
>
> - 스케일 < 1: 기울기 **소실** 예상 (매번 1보다 작은 값을 곱하니까)
> - 스케일 = 1: **경계** (소실도 폭발도 아닌 임계점)
> - 스케일 > 1: 기울기 **폭발** 예상 (매번 1보다 큰 값을 곱하니까)

```
W_hh 스케일별 기울기 변화 예상:

기울기 크기
    ↑
    │  ╱ 스케일 1.5 (폭발!)
    │ ╱
    │╱──── 스케일 1.0 (경계)
    │╲
    │ ╲── 스케일 0.9 (서서히 소실)
    │  ╲
    │   ╲─ 스케일 0.5 (빠르게 소실)
    └────────────────────→ 시점 (과거 ← 현재)
          t=0  ...  t=50
```

In [ ]:
# ============================================================
# 2.5 기울기 소실 시각화
# 긴 시퀀스에서 BPTT의 기울기가 어떻게 변하는지 관찰
# ============================================================

np.random.seed(42)  # 재현성

# --- 긴 시퀀스로 기울기 소실을 명확히 관찰 ---
long_seq_len = 50  # 50 시점 (충분히 길어야 기울기 소실이 보임)
d_vis, h_vis, o_vis = 1, 8, 1  # 작은 차원

# --- 다양한 W_hh 스케일로 기울기 소실/폭발 비교 ---
# W_hh의 크기(스케일)에 따라 기울기 소실/폭발이 달라짐
scales = [0.5, 0.9, 1.0, 1.5]  # W_hh에 곱할 스케일 팩터
scale_labels = ['W_hh x 0.5 (소실)', 'W_hh x 0.9 (약한 소실)',
                'W_hh x 1.0 (경계)', 'W_hh x 1.5 (폭발)']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for scale, label in zip(scales, scale_labels):
    # 새 RNN 생성
    rnn_vis = RNNCell(d_vis, h_vis, o_vis)
    # W_hh의 크기를 조절하여 기울기 소실/폭발을 제어
    rnn_vis.W_hh *= scale

    # 랜덤 입력 시퀀스 생성
    inputs_vis = [np.random.randn(d_vis, 1) * 0.5 for _ in range(long_seq_len)]
    targets_vis = [np.random.randn(o_vis, 1) for _ in range(long_seq_len)]

    # 순전파
    outputs_vis, _ = rnn_vis.forward(inputs_vis)

    # 손실 기울기 계산
    d_outputs_vis = [outputs_vis[t] - targets_vis[t] for t in range(long_seq_len)]

    # 역전파 (기울기만 계산, 가중치 업데이트 없음)
    grad_norms = rnn_vis.backward(d_outputs_vis, learning_rate=0.0)

    # 왼쪽: 원래 스케일
    axes[0].plot(range(long_seq_len), grad_norms, linewidth=1.5, label=label)
    # 오른쪽: 로그 스케일 (지수적 변화를 선형으로 보기 위해)
    axes[1].plot(range(long_seq_len), grad_norms, linewidth=1.5, label=label)

# --- 왼쪽 그래프: 원래 스케일 ---
axes[0].set_title('시점별 기울기 크기 (원래 스케일)', fontsize=14)
axes[0].set_xlabel('시점 (t)')
axes[0].set_ylabel('기울기 크기 ||dL/da_t||')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# --- 오른쪽 그래프: 로그 스케일 ---
axes[1].set_title('시점별 기울기 크기 (로그 스케일)', fontsize=14)
axes[1].set_xlabel('시점 (t)')
axes[1].set_ylabel('기울기 크기 (log)')
axes[1].set_yscale('log')  # y축을 로그 스케일로
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("관찰 포인트:")
print("1. W_hh 스케일이 1보다 작으면 → 기울기가 초기 시점(왼쪽)으로 갈수록 급격히 감소 (기울기 소실)")
print("2. W_hh 스케일이 1보다 크면 → 기울기가 초기 시점으로 갈수록 급격히 증가 (기울기 폭발)")
print("3. 로그 스케일에서 직선 = 지수적 변화 → Part 1.4의 수식 λ^(t-k)와 일치!")

---

# Part 3: PyTorch nn.RNN 사용

---

## 3.1 nn.RNN 파라미터 상세 설명

PyTorch의 `nn.RNN`은 우리가 Part 2에서 구현한 것과 **동일한 수식**을 사용하지만, 최적화되고 배치(batch, 여러 데이터를 한번에) 처리를 지원합니다.

> **PyTorch를 쓰는 이유**: 우리가 Part 2에서 만든 RNNCell은 교육용이고, PyTorch의 nn.RNN은 GPU 가속, 자동 미분(autograd), 배치 처리 등을 지원하여 실제 프로젝트에서 사용합니다.

```python
nn.RNN(input_size, hidden_size, num_layers=1, batch_first=False, ...)
```

| 파라미터 | 의미 | 우리 구현에서 |
|----------|------|--------------|
| `input_size` | 입력 벡터 차원 $d$ | `input_size` |
| `hidden_size` | 은닉 상태 차원 $h$ | `hidden_size` |
| `num_layers` | RNN 층 수 (쌓기) | 1층만 구현 |
| `nonlinearity` | 활성화 함수 (`'tanh'` 또는 `'relu'`) | `tanh` 고정 |
| `batch_first` | `True`면 입력 shape = (배치, 시퀀스, 특성) | 미지원 |
| `dropout` | 층 사이 드롭아웃(dropout, 무작위 비활성화) 비율 | 미지원 |
| `bidirectional` | 양방향(bidirectional) RNN 여부 | 미지원 |

### nn.RNN의 입출력

```python
output, h_n = rnn(input, h_0)
```

| 변수 | Shape | 의미 |
|------|-------|------|
| `input` | `(T, batch, d)` 또는 `(batch, T, d)` if batch_first | 입력 시퀀스 |
| `h_0` | `(num_layers, batch, h)` | 초기 은닉 상태 |
| `output` | `(T, batch, h)` 또는 `(batch, T, h)` | **모든 시점**의 은닉 상태 |
| `h_n` | `(num_layers, batch, h)` | **마지막 시점**의 은닉 상태 |

> **주의**: `output`은 $y_t$가 아니라 $h_t$입니다! 출력층($W_{hy}$)은 별도로 추가해야 합니다. 이것이 우리 구현과 다른 점입니다.

### 우리 구현 vs PyTorch nn.RNN 비교

```
우리 구현 (RNNCell):                   PyTorch (nn.RNN):
┌────────────────────┐               ┌────────────────────┐
│ forward(inputs)     │               │ rnn(input, h_0)     │
│  → outputs (y_t)    │               │  → output (h_t)     │ ← 주의!
│  → hiddens (h_t)    │               │  → h_n (마지막 h)   │
│                      │               │                      │
│ 출력층 포함 (W_hy)  │               │ 출력층 미포함!       │
│ 배치 미지원          │               │ 배치 지원            │
│ CPU only             │               │ GPU 지원             │
└────────────────────┘               └────────────────────┘
```

In [ ]:
# ============================================================
# 3.1 PyTorch nn.RNN 기본 사용법 확인
# ============================================================

import torch              # PyTorch 메인 모듈
import torch.nn as nn     # 신경망 모듈 (nn.RNN이 여기에 있음)

# --- nn.RNN 생성 ---
# 우리가 Part 2에서 만든 것과 동일한 차원 설정
input_size_pt = 2    # 입력 차원 d = 2
hidden_size_pt = 3   # 은닉 차원 h = 3
num_layers = 1       # RNN 층 수 = 1 (단층)

# nn.RNN 생성
# batch_first=True: 입력 shape를 (배치, 시퀀스, 특성) 순서로 사용
# (왜? 직관적으로 "N개의 시퀀스, 각각 T시점, 각 시점에서 d차원"이라고 읽기 쉬움)
rnn_pt = nn.RNN(
    input_size=input_size_pt,    # 각 시점 입력의 차원
    hidden_size=hidden_size_pt,  # 은닉 상태의 차원
    num_layers=num_layers,       # RNN 층 수
    nonlinearity='tanh',         # 활성화 함수 (기본값: tanh)
    batch_first=True             # 입력 shape: (배치, 시퀀스, 특성)
)

# --- nn.RNN의 내부 파라미터 확인 ---
print("nn.RNN의 학습 가능한 파라미터:")
print("=" * 60)
for name, param in rnn_pt.named_parameters():
    print(f"  {name:20s} | shape: {str(param.shape):15s} | 원소 수: {param.numel()}")

print(f"\n총 학습 가능한 파라미터 수: {sum(p.numel() for p in rnn_pt.parameters())}")

# --- 파라미터 이름 해석 ---
print("\n파라미터 이름 해석:")
print("  weight_ih_l0 = W_xh (입력→은닉), 'ih' = input-hidden, 'l0' = layer 0")
print("  weight_hh_l0 = W_hh (은닉→은닉), 'hh' = hidden-hidden")
print("  bias_ih_l0   = b_h의 일부 (입력 편향)")
print("  bias_hh_l0   = b_h의 일부 (은닉 편향)")
print("\n참고: PyTorch는 b_h를 두 개로 나누어 저장합니다 (bias_ih + bias_hh)")
print("      실제 계산: h_t = tanh(W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh)")

In [ ]:
# ============================================================
# nn.RNN 순전파 실행 예제
# ============================================================

# --- 더미 입력 데이터 생성 ---
batch_size = 2   # 배치 크기: 2개의 시퀀스를 동시에 처리
seq_length = 4   # 시퀀스 길이: 4 시점

# 입력 텐서 생성: shape = (배치, 시퀀스, 특성) = (2, 4, 2)
# batch_first=True이므로 배치가 첫 번째 차원
x_dummy = torch.randn(batch_size, seq_length, input_size_pt)
print(f"입력 shape: {x_dummy.shape}  → (배치={batch_size}, 시퀀스={seq_length}, 특성={input_size_pt})")

# 초기 은닉 상태: shape = (층수, 배치, 은닉) = (1, 2, 3)
h0 = torch.zeros(num_layers, batch_size, hidden_size_pt)
print(f"초기 은닉 h0 shape: {h0.shape}  → (층수={num_layers}, 배치={batch_size}, 은닉={hidden_size_pt})")

# --- 순전파 실행 ---
output_pt, h_n = rnn_pt(x_dummy, h0)

print(f"\n출력 shape: {output_pt.shape}")
print(f"  → (배치={batch_size}, 시퀀스={seq_length}, 은닉={hidden_size_pt})")
print(f"  → 모든 시점의 은닉 상태 h_0, h_1, h_2, h_3가 담겨있음")

print(f"\n마지막 은닉 h_n shape: {h_n.shape}")
print(f"  → (층수={num_layers}, 배치={batch_size}, 은닉={hidden_size_pt})")
print(f"  → 마지막 시점의 은닉 상태 h_3만 담겨있음")

# --- output의 마지막 시점 = h_n 인지 확인 ---
# output[:, -1, :] (마지막 시점)과 h_n[0] (마지막 층의 최종 은닉)이 같아야 함
print(f"\noutput[:, -1, :] == h_n[0] ? {torch.allclose(output_pt[:, -1, :], h_n[0])}")
print("→ 맞습니다! output의 마지막 시점이 곧 h_n입니다.")

## 3.2 감정 분류 예제 (Many-to-One, 다대일)

간단한 감정 분류(긍정/부정) 문제를 PyTorch `nn.RNN`으로 구현합니다.

- **구조**: Many-to-One (시퀀스 전체를 읽고 마지막에 분류)
- **데이터**: 간단한 합성 데이터 (양수 합계 → 긍정, 음수 합계 → 부정)
- **흐름**: 입력 시퀀스 → RNN → 마지막 은닉 상태 → 분류 헤드(head) → 확률

> **왜 Many-to-One?**: 감정 분류는 "문장 전체를 다 읽은 뒤에" 판단해야 합니다. 예를 들어 "이 영화는 지루하지만 결말이 대박이다"는 전체를 읽어야 긍정인지 부정인지 알 수 있죠.

### 모델 구조

```
시퀀스 입력:                          분류 출력:
x₀   x₁   x₂  ...  x₉
 ↓    ↓    ↓         ↓
[h₀]→[h₁]→[h₂]→...→[h₉]            ← RNN: 시퀀스를 읽으며 기억 누적
                       ↓
                  [Linear]            ← 분류 헤드: 은닉→클래스
                       ↓
                   긍정/부정           ← 최종 출력
```

> **합성 데이터를 쓰는 이유**: 실제 텍스트 데이터는 전처리(토큰화, 임베딩 등)가 복잡합니다. 여기서는 RNN의 동작 원리에 집중하기 위해 단순한 숫자 데이터를 사용합니다.

In [ ]:
# ============================================================
# 3.2 감정 분류 데이터 생성
# 규칙: 시퀀스의 합이 양수면 긍정(1), 음수면 부정(0)
# ============================================================

torch.manual_seed(42)  # PyTorch 난수 시드 고정

# --- 합성 데이터 생성 ---
num_samples = 1000    # 총 샘플 수
seq_len_cls = 10      # 시퀀스 길이: 10 시점
feature_dim = 1       # 입력 특성 차원: 1 (스칼라)

# 랜덤 시퀀스 생성: shape = (1000, 10, 1)
# 각 시점에서 -1~1 사이의 랜덤 값
X_cls = torch.randn(num_samples, seq_len_cls, feature_dim)

# 라벨 생성: 시퀀스의 합이 양수면 1(긍정), 음수면 0(부정)
# 시퀀스 전체를 "읽어야" 합을 알 수 있으므로, RNN에 적합한 문제!
Y_cls = (X_cls.sum(dim=1).squeeze() > 0).long()  # shape = (1000,)

# --- 학습/테스트 분할 ---
train_size = 800  # 800개로 학습
X_train = X_cls[:train_size]   # 학습 입력
Y_train = Y_cls[:train_size]   # 학습 라벨
X_test = X_cls[train_size:]    # 테스트 입력
Y_test = Y_cls[train_size:]    # 테스트 라벨

print(f"학습 데이터: {X_train.shape} -> 라벨: {Y_train.shape}")
print(f"테스트 데이터: {X_test.shape} -> 라벨: {Y_test.shape}")
print(f"\n라벨 분포 (학습):")
print(f"  긍정(1): {(Y_train == 1).sum().item()}개 ({(Y_train == 1).float().mean():.1%})")
print(f"  부정(0): {(Y_train == 0).sum().item()}개 ({(Y_train == 0).float().mean():.1%})")

# --- 예시 데이터 확인 ---
print(f"\n예시 시퀀스 (첫 번째):")
print(f"  값: {X_train[0].squeeze().numpy().round(3)}")
print(f"  합: {X_train[0].sum().item():.3f}")
print(f"  라벨: {Y_train[0].item()} ({'긍정' if Y_train[0].item() == 1 else '부정'})")

In [ ]:
# ============================================================
# 3.2 감정 분류 모델 정의 (Many-to-One RNN)
# ============================================================

class SentimentRNN(nn.Module):
    """
    Many-to-One 구조의 RNN 감정 분류기.

    구조:
        입력 시퀀스 -> nn.RNN -> 마지막 은닉 상태 -> Linear(분류 헤드) -> 2클래스 확률

    수식으로 보면:
        h_t = tanh(W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh)  (RNN 내부)
        logits = W_hy @ h_{T-1} + b_y                            (분류 헤드)
    """

    def __init__(self, input_size: int, hidden_size: int, num_classes: int):
        """
        Args:
            input_size: 입력 특성 차원 d
            hidden_size: 은닉 상태 차원 h
            num_classes: 분류 클래스 수 (2: 긍정/부정)
        """
        super().__init__()  # nn.Module 초기화 (필수)

        self.hidden_size = hidden_size  # 은닉 차원 저장 (forward에서 사용)

        # nn.RNN: PyTorch가 제공하는 최적화된 RNN
        # batch_first=True -> 입력 shape: (배치, 시퀀스, 특성)
        self.rnn = nn.RNN(
            input_size=input_size,     # 입력 차원 d
            hidden_size=hidden_size,   # 은닉 차원 h
            num_layers=1,              # 1층 RNN
            batch_first=True,          # 배치가 첫 번째 차원
            nonlinearity='tanh'        # Part 1에서 배운 tanh 활성화
        )

        # 분류 헤드: 은닉 상태 -> 클래스 로짓
        # 이것이 수식 2의 W_hy @ h_t + b_y에 해당
        self.fc = nn.Linear(hidden_size, num_classes)  # (h) -> (num_classes)

    def forward(self, x):
        """
        순전파: 시퀀스를 읽고 분류 결과를 출력합니다.

        Args:
            x: 입력 시퀀스, shape = (배치, 시퀀스길이, 특성)

        Returns:
            logits: 분류 로짓, shape = (배치, num_classes)
        """
        batch_size = x.size(0)  # 배치 크기

        # 초기 은닉 상태: 영벡터로 시작
        # shape = (1, 배치, 은닉) -- 1은 num_layers
        h0 = torch.zeros(1, batch_size, self.hidden_size)

        # --- RNN 순전파 ---
        # output: 모든 시점의 은닉 상태, shape = (배치, 시퀀스, 은닉)
        # h_n: 마지막 시점의 은닉 상태, shape = (1, 배치, 은닉)
        output, h_n = self.rnn(x, h0)

        # --- Many-to-One: 마지막 시점의 은닉 상태만 사용 ---
        # h_n[0]: shape = (배치, 은닉) -- 층 차원을 제거
        # 또는 output[:, -1, :] 와 동일
        last_hidden = h_n[0]  # (배치, 은닉)

        # --- 분류 헤드 ---
        # W_hy @ h_{T-1} + b_y -> (배치, num_classes)
        logits = self.fc(last_hidden)

        return logits  # softmax는 CrossEntropyLoss 내부에서 자동 적용

# --- 모델 생성 ---
model = SentimentRNN(
    input_size=feature_dim,    # 1 (스칼라 입력)
    hidden_size=32,            # 은닉 차원 32
    num_classes=2              # 긍정/부정 2클래스
)

print("모델 구조:")
print(model)
print(f"\n총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# 3.2 감정 분류 모델 학습
# ============================================================

# --- 손실 함수 및 옵티마이저 설정 ---
# CrossEntropyLoss: 분류 문제의 표준 손실 함수
# 내부에서 softmax + 음의 로그 우도(NLL)를 함께 계산
criterion = nn.CrossEntropyLoss()

# Adam 옵티마이저: SGD보다 빠르고 안정적인 최적화 알고리즘
# lr=0.01: 학습률 -- RNN은 학습이 불안정할 수 있으므로 적절히 설정
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- 학습 루프 ---
num_epochs_pt = 50       # 전체 데이터를 50번 반복
batch_size_train = 64    # 미니배치 크기: 64개씩 묶어서 학습
train_losses = []        # 에포크별 손실 기록
train_accs = []          # 에포크별 정확도 기록

for epoch in range(num_epochs_pt):
    model.train()  # 학습 모드 (드롭아웃 등이 활성화됨)
    epoch_loss = 0     # 이 에포크의 총 손실
    epoch_correct = 0  # 이 에포크의 정답 수

    # --- 미니배치 학습 ---
    for i in range(0, train_size, batch_size_train):
        # 미니배치 추출
        x_batch = X_train[i:i+batch_size_train]  # 입력 배치
        y_batch = Y_train[i:i+batch_size_train]  # 라벨 배치

        # 순전파: 모델에 입력을 넣어 예측값 계산
        logits = model(x_batch)  # (배치, 2) -- 2클래스 로짓

        # 손실 계산: 예측과 정답의 차이
        loss = criterion(logits, y_batch)

        # 역전파: 기울기 계산
        optimizer.zero_grad()  # 이전 기울기 초기화 (중요! 안 하면 기울기가 누적됨)
        loss.backward()        # 역전파로 기울기 계산 (PyTorch가 자동으로 BPTT 수행)

        # 기울기 클리핑: RNN의 기울기 폭발 방지
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 가중치 업데이트: 계산된 기울기 방향으로 가중치 수정
        optimizer.step()

        # --- 통계 기록 ---
        epoch_loss += loss.item() * len(x_batch)  # 배치 손실 누적
        # argmax: 로짓에서 가장 큰 값의 인덱스 = 예측 클래스
        predictions = logits.argmax(dim=1)
        epoch_correct += (predictions == y_batch).sum().item()  # 정답 수 누적

    # 에포크 평균 계산
    avg_loss = epoch_loss / train_size
    avg_acc = epoch_correct / train_size
    train_losses.append(avg_loss)
    train_accs.append(avg_acc)

    # 10 에포크마다 진행 상황 출력
    if (epoch + 1) % 10 == 0:
        print(f"에포크 {epoch+1:3d}/{num_epochs_pt} | "
              f"손실: {avg_loss:.4f} | 정확도: {avg_acc:.1%}")

print("\n학습 완료!")

In [ ]:
# ============================================================
# 3.2 학습 결과 시각화 + 테스트 평가
# ============================================================

# --- 학습 곡선 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 손실 곡선
axes[0].plot(train_losses, 'b-', linewidth=1.5)
axes[0].set_title('PyTorch RNN 학습 손실 곡선', fontsize=14)
axes[0].set_xlabel('에포크')
axes[0].set_ylabel('CrossEntropy 손실')
axes[0].grid(True, alpha=0.3)

# 오른쪽: 정확도 곡선
axes[1].plot(train_accs, 'g-', linewidth=1.5)
axes[1].set_title('PyTorch RNN 학습 정확도', fontsize=14)
axes[1].set_xlabel('에포크')
axes[1].set_ylabel('정확도')
axes[1].set_ylim([0.4, 1.05])  # y축 범위 고정
axes[1].axhline(y=0.5, color='r', linestyle='--', label='랜덤 기준선 (50%)')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- 테스트 세트 평가 ---
model.eval()  # 평가 모드 (드롭아웃 비활성화)
with torch.no_grad():  # 기울기 계산 비활성화 (메모리 절약, 속도 향상)
    # 테스트 데이터 전체를 한번에 순전파
    test_logits = model(X_test)
    test_preds = test_logits.argmax(dim=1)  # 예측 클래스
    test_acc = (test_preds == Y_test).float().mean().item()  # 정확도 계산

print(f"\n테스트 정확도: {test_acc:.1%}")
print(f"랜덤 기준선: 50.0% (이진 분류)")
print(f"향상도: +{(test_acc - 0.5)*100:.1f}%p")

# --- 예측 예시 ---
print(f"\n예측 예시 (처음 5개):")
for i in range(5):
    seq_sum = X_test[i].sum().item()
    pred = test_preds[i].item()
    true = Y_test[i].item()
    correct = "O" if pred == true else "X"
    print(f"  시퀀스 합: {seq_sum:+.3f} → 예측: {pred}({'긍정' if pred else '부정'}) | "
          f"정답: {true}({'긍정' if true else '부정'}) [{correct}]")

---

# Part 4: 정리

---

## 4.1 RNN의 한계와 LSTM으로의 발전 동기

### Vanilla RNN의 근본적 한계

이 노트북에서 직접 구현하고 시각화한 것처럼, Vanilla RNN에는 심각한 한계가 있습니다:

| 한계 | 원인 | 결과 | 비유 |
|------|------|------|------|
| **기울기 소실** | $\prod_{j} (1-h_j^2) \cdot W_{hh}$의 반복 곱 | 20~30 시점 이상의 장기 의존성 학습 불가 | 전화기 게임: 멀리 전달될수록 메시지가 흐려짐 |
| **기울기 폭발** | $W_{hh}$의 특이값 > 1 | 학습 불안정, NaN 발생 | 마이크 하울링: 작은 소리가 증폭되어 귀가 아픔 |
| **단기 기억** | 은닉 상태에 새 정보가 덮어쓰여짐 | 먼 과거 정보가 희석됨 | 칠판: 계속 새로 쓰면 이전 내용이 지워짐 |

### LSTM이 이 문제를 어떻게 해결하는가?

LSTM(Long Short-Term Memory, 장단기 기억)은 **3개의 게이트(gate, 문)**와 **셀 상태(Cell State, 장기 기억 저장소)**를 추가하여 기울기 소실을 해결합니다:

> **LSTM을 쉽게 말하면**: Vanilla RNN이 "칠판 하나"에 모든 걸 기록하는 것이라면, LSTM은 "칠판 + 노트북"을 함께 쓰는 것입니다. 노트북(셀 상태)에는 장기 기억을, 칠판(은닉 상태)에는 단기 기억을 따로 관리합니다.

```
          ┌─────────────────── 셀 상태 (장기 기억 고속도로) ──────────────┐
          │        ×(잊기)         +(기억)                                 │
          ▼                                                                ▼
 ┌──────────────┐   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
 │  Forget Gate │   │  Input Gate  │   │   Cell State  │   │ Output Gate  │
 │  (망각 게이트)│   │  (입력 게이트)│   │   (셀 상태)   │   │ (출력 게이트)│
 │   f_t (σ)    │   │   i_t (σ)    │   │     c_t       │   │   o_t (σ)    │
 └──────────────┘   └──────────────┘   └──────────────┘   └──────────────┘
```

| 구성 요소 | 역할 | RNN 대비 장점 |
|-----------|------|--------------|
| **망각 게이트** $f_t$ | 이전 셀 상태를 얼마나 잊을지 결정 | 선택적 망각 가능 |
| **입력 게이트** $i_t$ | 새 정보를 얼마나 기억할지 결정 | 선택적 기억 가능 |
| **셀 상태** $c_t$ | 장기 기억 전용 경로 | **기울기가 직선 경로로 전파 → 소실 방지** |
| **출력 게이트** $o_t$ | 셀 상태의 어느 부분을 출력할지 결정 | 필요한 정보만 노출 |

> **핵심**: LSTM의 셀 상태는 **덧셈**으로 업데이트되므로, 기울기가 곱셈으로 감쇄되지 않고 직선 경로로 전파됩니다. 이것이 장기 의존성을 학습할 수 있는 이유입니다.

## 4.2 핵심 수식 요약표

이 노트북에서 다룬 모든 핵심 수식을 한 곳에 정리합니다.

### RNN 순전파

| 단계 | 수식 | 쉬운 설명 |
|------|------|-----------|
| 은닉 상태 | $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$ | 현재 입력 + 이전 기억 → 새 기억 |
| 출력 | $y_t = W_{hy} h_t + b_y$ | 기억을 바탕으로 답 출력 |
| 초기 상태 | $h_{-1} = \mathbf{0}$ | 기억이 없는 상태에서 시작 |

### 차원 정리

| 기호 | 차원 | 이름 | 비유 |
|------|------|------|------|
| $x_t$ | $(d, 1)$ | 입력 벡터 | 오늘 받은 정보 |
| $h_t$ | $(h, 1)$ | 은닉 상태 | 지금까지의 기억 |
| $y_t$ | $(o, 1)$ | 출력 벡터 | 현재 판단/예측 |
| $W_{xh}$ | $(h, d)$ | 입력→은닉 가중치 | 새 정보를 기억으로 변환 |
| $W_{hh}$ | $(h, h)$ | 은닉→은닉 가중치 | 이전 기억을 현재로 전달 |
| $W_{hy}$ | $(o, h)$ | 은닉→출력 가중치 | 기억을 출력으로 변환 |

### BPTT 역전파 (시간을 거슬러 올라가는 역전파)

| 단계 | 수식 | 쉬운 설명 |
|------|------|-----------|
| 출력 기울기 | $\frac{\partial L_t}{\partial W_{hy}} = \frac{\partial L_t}{\partial y_t} \cdot h_t^T$ | 출력층의 오차 → 가중치 수정량 |
| 은닉 기울기 | $\frac{\partial L}{\partial h_t} = W_{hy}^T \frac{\partial L_t}{\partial y_t} + W_{hh}^T \cdot \text{diag}(1-h_{t+1}^2) \cdot \frac{\partial L}{\partial h_{t+1}}$ | 현재 오차 + 미래에서 온 오차 |
| tanh 역전파 | $\frac{\partial h_t}{\partial a_t} = 1 - h_t^2$ | tanh의 기울기 (최대 1) |
| 기울기 전파 | $\frac{\partial h_{t+1}}{\partial h_t} = \text{diag}(1-h_{t+1}^2) \cdot W_{hh}$ | 시점 간 기울기 전달 |

### 기울기 소실/폭발

$$
\left\|\frac{\partial h_t}{\partial h_k}\right\| = \left\|\prod_{j=k+1}^{t} \text{diag}(1-h_j^2) \cdot W_{hh}\right\| \approx \lambda_{\max}(W_{hh})^{t-k}
$$

- $\lambda_{\max} < 1$: 기울기 소실 (Vanishing) → 먼 과거를 학습 못함
- $\lambda_{\max} > 1$: 기울기 폭발 (Exploding) → 학습이 불안정
- **대책**: 기울기 클리핑(clipping), LSTM/GRU 사용, 직교 초기화(orthogonal init)

---

### 이 노트북에서 배운 것 요약

```
┌─────────────────────────────────────────────────────┐
│                    학습 체크리스트                     │
├─────────────────────────────────────────────────────┤
│ [Part 1] 이론                                        │
│   □ RNN이 필요한 이유 (시퀀스 데이터, 기억)          │
│   □ 핵심 수식 2개와 각 변수의 차원                   │
│   □ BPTT의 원리와 기울기 소실/폭발                   │
│                                                       │
│ [Part 2] NumPy 구현                                  │
│   □ RNNCell의 forward/backward 구현                  │
│   □ 수치 기울기로 역전파 검증                         │
│   □ 사인파 예측으로 실제 학습 확인                    │
│   □ 기울기 소실을 시각적으로 확인                     │
│                                                       │
│ [Part 3] PyTorch                                     │
│   □ nn.RNN의 입출력 구조 이해                        │
│   □ Many-to-One 감정 분류 구현                       │
└─────────────────────────────────────────────────────┘
```

### 다음 단계

이 노트북에서 Vanilla RNN의 원리를 완전히 이해했다면, 다음으로 학습할 내용:

1. **LSTM**: 기울기 소실을 해결하는 게이트(gate) 메커니즘
2. **GRU**: LSTM의 경량 버전 (게이트 2개로 단순화)
3. **Attention(어텐션)**: 시퀀스의 중요한 부분에 집중하는 메커니즘
4. **Transformer(트랜스포머)**: Attention만으로 시퀀스를 처리 (RNN 없이!)

> "RNN을 밑바닥부터 이해했다면, LSTM/GRU/Transformer는 그 위에 세우는 건물일 뿐입니다."
>
> 다음 노트북에서 LSTM을 구현할 때, 이 노트북의 `RNNCell`을 확장하여 게이트를 추가해보겠습니다.